# 实验配置、日志与可复现记录

## 学习目标

把一次训练实验的模型、数据、超参数、随机种子、指标和 checkpoint 记录在一起，使结果可以解释、比较和复现。

## 概念模型

实验结果不是一个 accuracy 数字，而是 `(代码版本, 数据版本, 配置, 环境, checkpoint, 指标)` 的组合。配置和结果应在训练开始时固定并保存。

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import json
import torch
from torch import nn

@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 42
    batch_size: int = 8
    learning_rate: float = 0.01
    epochs: int = 2
    model_name: str = 'mlp-demo'

config = ExperimentConfig()
run_dir = Path('artifacts/notebook19-run')
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / 'config.json').write_text(json.dumps(asdict(config), indent=2), encoding='utf-8')
print(asdict(config))
assert json.loads((run_dir / 'config.json').read_text())['seed'] == 42

### 实验 1：记录配置、训练指标和运行元数据

**实验目的**：把超参数配置序列化，并按 epoch 保存结构化训练记录。配置、代码版本、依赖版本、设备、参数量和随机种子共同定义一次可追溯实验。

Notebook 输出不适合作为唯一日志；JSON 等结构化记录便于比较、绘图和自动检查。写入 run 目录时应避免覆盖已有实验，并保存异常/中断状态。


In [ ]:
model = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
x = torch.randn(32, 3)
y = (x.sum(dim=1) > 0).long()
history = []
for epoch in range(1, config.epochs + 1):
    optimizer.zero_grad(set_to_none=True)
    loss = nn.CrossEntropyLoss()(model(x), y)
    loss.backward(); optimizer.step()
    record = {'epoch': epoch, 'train_loss': float(loss.item()), 'lr': optimizer.param_groups[0]['lr']}
    history.append(record)
    print(record)
(run_dir / 'metrics.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
assert len(history) == config.epochs

In [ ]:
parameter_count = sum(parameter.numel() for parameter in model.parameters())
metadata = {'torch_version': torch.__version__, 'parameter_count': parameter_count, 'device': 'cpu'}
(run_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('metadata:', metadata)
assert parameter_count > 0

## 检查点

说明为什么超参数、随机种子和环境版本必须与指标一起保存；指出哪些内容适合 JSON，哪些内容必须使用 checkpoint 保存。

## 试一试

复制本实验并只修改学习率，比较两个 run 目录中的 `config.json` 和 `metrics.json`；再将模型参数保存为 checkpoint。

## 常见错误与调试

只保存最终 accuracy、覆盖旧实验、配置与实际代码不一致、没有记录数据划分、把随机种子误认为完全确定性、只保存模型参数却无法恢复优化器状态。